In [1]:
%pip install pyarrow

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.3 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%%writefile ../src/config.py
"""Central place for paths and column names. Change names here, nowhere else."""
from pathlib import Path

ROOT = Path(__file__).resolve().parents[1]
RAW_PATH = ROOT / "data" / "raw" / "APL_Logistics.csv"
PROCESSED_PATH = ROOT / "data" / "processed" / "apl_clean.parquet"
MODELS_DIR = ROOT / "models"

RANDOM_STATE = 42
ENCODING = "latin-1"          # the file is not UTF-8

# Core columns
TARGET = "Late_delivery_risk"
REAL_DAYS = "Days for shipping (real)"
SCHED_DAYS = "Days for shipment (scheduled)"
DELIVERY_STATUS = "Delivery Status"
SHIP_MODE = "Shipping Mode"
MARKET = "Market"
REGION = "Order Region"
COUNTRY = "Order Country"
SEGMENT = "Customer Segment"

# Columns created by cleaning
GAP = "delay_gap"
DELIVERY_CLASS = "delivery_class"
IS_CANCELED = "is_canceled"

# Personal data (not needed for any analysis) and a duplicate ID column
PII_COLS = ["Customer Fname", "Customer Lname", "Customer Street", "Customer Zipcode"]
REDUNDANT_COLS = ["Order Customer Id"]   # identical to Customer Id in this file

VALID_SCHEDULED_DAYS = {0, 1, 2, 4}

Overwriting ../src/config.py


In [3]:
%%writefile ../src/data.py
"""Data loading and cleaning."""
import pandas as pd
from src import config


def load_raw() -> pd.DataFrame:
    """Read the raw CSV exactly as delivered, with no changes."""
    return pd.read_csv(config.RAW_PATH, encoding=config.ENCODING)


def clean(df_raw: pd.DataFrame):
    """Return (clean_df, report). Never modifies df_raw."""
    df = df_raw.copy()
    report = {"rows_in": len(df)}

    # 1. Standardise text: trim and collapse repeated spaces
    #    (e.g. 'South of  USA ' -> 'South of USA')
    text_cols = df.select_dtypes(include=["object", "string"]).columns
    for c in text_cols:
        df[c] = df[c].str.strip().str.replace(r"\s+", " ", regex=True)

    # 2. Validate shipping durations
    bad_real = df[config.REAL_DAYS].isna() | (df[config.REAL_DAYS] < 0)
    bad_sched = ~df[config.SCHED_DAYS].isin(config.VALID_SCHEDULED_DAYS)
    report["invalid_rows_dropped"] = int((bad_real | bad_sched).sum())
    df = df[~(bad_real | bad_sched)]

    # 3. Remove personal data and the duplicate ID column
    df = df.drop(columns=config.PII_COLS + config.REDUNDANT_COLS, errors="ignore")

    # 4. Delay gap and delivery class
    df[config.GAP] = df[config.REAL_DAYS] - df[config.SCHED_DAYS]
    df[config.DELIVERY_CLASS] = pd.cut(
        df[config.GAP], bins=[-99, -1, 0, 99],
        labels=["Early", "On-time", "Delayed"],
    ).astype("object")

    # 5. Canceled shipments never reached the customer: flag, do not delete
    df[config.IS_CANCELED] = df[config.DELIVERY_STATUS].eq("Shipping canceled")

    report["rows_out"] = len(df)
    report["canceled_flagged"] = int(df[config.IS_CANCELED].sum())
    return df.reset_index(drop=True), report


def delivered_only(df: pd.DataFrame) -> pd.DataFrame:
    """Rows that actually shipped. Use this for every delay analysis."""
    return df[~df[config.IS_CANCELED]]

Overwriting ../src/data.py


In [4]:
import sys, pandas as pd
sys.path.append("..")

from src import config
from src.data import load_raw, clean, delivered_only

raw = load_raw()
df, report = clean(raw)
print(report)
print("Clean shape:", df.shape)
print("Regions:", df[config.REGION].nunique())

df.to_parquet(config.PROCESSED_PATH, index=False)
print("Saved:", config.PROCESSED_PATH.exists())

{'rows_in': 180519, 'invalid_rows_dropped': 0, 'rows_out': 180519, 'canceled_flagged': 7754}
Clean shape: (180519, 38)
Regions: 23
Saved: True


In [5]:
print(pd.crosstab(df[config.DELIVERY_CLASS], df[config.IS_CANCELED]))

d = delivered_only(df)
print("\nDelivered rows:", len(d))
print("Target == (gap >= 1) on delivered rows:",
      (d[config.TARGET] == (d[config.GAP] >= 1).astype(int)).mean())

is_canceled     False  True 
delivery_class              
Delayed         98977   4423
Early           41592   1774
On-time         32196   1557

Delivered rows: 172765
Target == (gap >= 1) on delivered rows: 1.0


In [6]:
print(d[config.DELIVERY_CLASS].value_counts(normalize=True).round(4))
print("\nOn-time rate (gap <= 0):", round((d[config.GAP] <= 0).mean(), 4))
print("Late rate              :", round(d[config.TARGET].mean(), 4))
print("Average delay gap      :", round(d[config.GAP].mean(), 4))

(d.groupby(config.SHIP_MODE)
  .agg(n=(config.GAP, "size"),
       late_rate=(config.TARGET, "mean"),
       avg_gap=(config.GAP, "mean"))
  .round(4))

delivery_class
Delayed    0.5729
Early      0.2407
On-time    0.1864
Name: proportion, dtype: float64

On-time rate (gap <= 0): 0.4271
Late rate              : 0.5729
Average delay gap      : 0.5655


,n,late_rate,avg_gap
Shipping Mode,,,
First Class,26513,1.0000,1.0000
Same Day,9293,0.4793,0.4793
Second Class,33806,0.7983,1.9932
Standard Class,103153,0.3977,-0.0063
